# 3b · Find the apex of strokes you annotated by hand

**What this notebook does:** If you (or a colleague) have already marked gesture **strokes** by hand in ELAN, this
notebook finds the **apex** of each stroke automatically.

**Some gesture vocabulary.** A gesture is often split into phases: the **preparation** (the hand gets ready),
the **stroke** (the meaningful, effortful part), sometimes a **hold** (the hand stays still), and the
**retraction** (the hand goes back to rest). The **apex** is the peak moment of the stroke: the instant the
movement "hits" its target, where the hand typically brakes sharply.

**How the apex is found.** Inside each stroke we look for clearly fast moments: faster than the stroke's average
speed plus one standard deviation (a measure of how much the speed varies). Right after such a fast moment the hand
brakes, and the slowest point after it is taken as an apex. If a stroke has no clearly fast moment, its slowest
point is the apex.

| You need | You get |
|---|---|
| `Upsampled/Test_Video_processed_data.csv` (from Notebook 2) | `Apexes/Test_Video_apexes.csv`: the speed data inside each stroke, with `AX` in the `apex` column at each apex. Notebook 5 adds them to ELAN as an `apex` tier. |
| An ELAN file with a tier of stroke annotations (see below) | |

### Before you start: annotate a few strokes in ELAN

The workshop files have no hand-made stroke annotations, so make a few yourself. It takes about 5 minutes:

1. Start **ELAN** and open `Elan/Test_Video.eaf` (*File → Open…*). If ELAN can't find the video, point it to
   `Videos/Test_Video.MOV`.
2. Add a tier: *Tier → Add New Tier…*, type `Stroke` as the *Tier Name*, click *Add*, then *Close*.
3. Mark a stroke: in the timeline at the bottom, drag across the stretch of time where a stroke happens so that it
   is highlighted. Then choose *Annotation → New Annotation Here* with the `Stroke` tier selected (or double-click
   on the `Stroke` tier inside the highlighted area), type `S` and press Enter.
4. Repeat this for a few strokes, then save (*File → Save*) and close ELAN.

**Already have stroke annotations?** Put your `.eaf` file in the `Elan` folder and set `ELAN_FILE`,
`STROKE_TIER` and `STROKE_LABELS` in the Settings below to match it.

## Step 1 · Load the tools

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pympi import Elan                # reads ELAN (.eaf) files
from scipy.signal import find_peaks

## Step 2 · Settings

In [ ]:
VIDEO_NAME = "Test_Video"

# Whose speed to use. It must be one of the KEYPOINTS you chose in Notebook 2.
KEYPOINT = "right_wrist"

# The ELAN file with your stroke annotations, and the name of the tier they are on
ELAN_FILE = f"../Elan/{VIDEO_NAME}.eaf"
STROKE_TIER = "Stroke"

# Which annotation texts count as strokes.
# Write None (without quotes) to use every annotation on the tier.
STROKE_LABELS = ["S", "Stroke"]

# Add this many milliseconds (one video frame) to the end of every stroke,
# so that its last frame is included.
EXTEND_END_MS = 33

## Step 3 · Load the speed data and the strokes

In [ ]:
data = pd.read_csv(f"../Upsampled/{VIDEO_NAME}_processed_data.csv")
speed_column = f"{KEYPOINT}_speed"

if not Path(ELAN_FILE).exists():
    raise FileNotFoundError(f"Can't find {ELAN_FILE}. Check ELAN_FILE in the Settings.")
elan = Elan.Eaf(ELAN_FILE)
if STROKE_TIER not in elan.get_tier_names():
    raise ValueError(f"There is no tier called '{STROKE_TIER}' in {ELAN_FILE}. "
                     f"The tiers in this file are: {list(elan.get_tier_names())}. "
                     "See 'Before you start' at the top of this notebook.")

strokes = pd.DataFrame(elan.get_annotation_data_for_tier(STROKE_TIER),
                       columns=["start_time", "end_time", "phase"])
if STROKE_LABELS is not None:
    strokes = strokes[strokes["phase"].isin(STROKE_LABELS)]
strokes = strokes.sort_values("start_time").reset_index(drop=True)
strokes["end_time"] += EXTEND_END_MS
strokes.insert(0, "gesture_id", range(1, len(strokes) + 1))

print(f"Found {len(strokes)} strokes on the '{STROKE_TIER}' tier")
strokes

## Step 4 · Find the apex of each stroke

For each stroke we mark the 10 ms measurements that fall inside it, then look for its apex as described at the top.

In [ ]:
data["phase"] = None
data["gesture_id"] = np.nan
data["apex"] = None

for stroke in strokes.itertuples():
    inside = (data["time_ms"] >= stroke.start_time) & (data["time_ms"] <= stroke.end_time)
    data.loc[inside, "phase"] = stroke.phase
    data.loc[inside, "gesture_id"] = stroke.gesture_id

    speed = data.loc[inside, speed_column].to_numpy()
    if len(speed) < 2:
        continue  # stroke too short to say anything

    clearly_fast = speed.mean() + speed.std()
    peaks, _ = find_peaks(speed, height=clearly_fast)
    if len(peaks) > 0:
        # after each fast moment: the slowest point from there to the end of the stroke
        apexes = [peak + np.argmin(speed[peak:]) for peak in peaks]
    else:
        apexes = [np.argmin(speed)]  # no clearly fast moment: the slowest point of the stroke

    rows = data.index[inside]
    data.loc[rows[apexes], "apex"] = "AX"

in_strokes = data[data["phase"].notna()]
in_strokes = in_strokes[["time_ms", f"{KEYPOINT}_x", f"{KEYPOINT}_y", speed_column,
                         "phase", "gesture_id", "apex"]]
print(f"Found {in_strokes['apex'].notna().sum()} apexes in {len(strokes)} strokes")
in_strokes[in_strokes["apex"].notna()]

## Step 5 · Save

In [ ]:
Path("../Apexes").mkdir(exist_ok=True)
output_path = f"../Apexes/{VIDEO_NAME}_apexes.csv"
in_strokes.to_csv(output_path, index=False)
print("Saved:", output_path)

## Step 6 · Check the result

Green areas are your strokes, red dots the apexes found. Use the slider to move through the video.

In [ ]:
apexes = in_strokes[in_strokes["apex"].notna()]

fig = go.Figure()
fig.add_scatter(x=data["time_ms"], y=data[speed_column], name="speed", line_color="black")
fig.add_scatter(x=apexes["time_ms"], y=apexes[speed_column], name="apex", mode="markers",
                marker=dict(color="red", size=10))
for stroke in strokes.itertuples():
    fig.add_vrect(x0=stroke.start_time, x1=stroke.end_time, fillcolor="green", opacity=0.2,
                  line_width=0)
fig.update_layout(title="Strokes (green) and their apexes (red)", xaxis_title="Time (ms)",
                  yaxis_title="Speed (1 = fastest)", xaxis_rangeslider_visible=True)
fig.show()

## Done!

Continue with **Notebook 5**: it adds the apexes to the ELAN file as an `apex` tier (next to the tiers from
Notebooks 3a and 4, if you ran those too).